## Serialize a Raster Data using SchemaOrg and the CUAHSI.org-Raster extension

The purpose of this notebook is to evaluate how Raster data can be extracted and mapped to our Pydantic classes.

In [1]:
import os
import sys
import hashlib
import rasterio
import mimetypes
from glob import glob
from pyproj import CRS
from pathlib import Path

# add the parent directory to the path. This is the 
# directory that contains our pydantic classes.
sys.path.append('..')
import base
import core
import dataset
import raster
import datavariable

In [2]:
def compute_sha256(file_path: Path) -> str:
    """Computes the SHA256 hash of a file.

    Args:
        file_path: The path to the file.

    Returns:
        The hexadecimal representation of the SHA256 hash.
    """
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            sha256_hash.update(chunk)
    return sha256_hash.hexdigest()

In [3]:
# Add raster MIME types if not already present
mimetypes.add_type("image/tiff", ".tif")
mimetypes.add_type("text/plain", ".asc")

In [8]:

def encode_raster_metadata(filepath, validate_bbox=True):
    # Open the Raster file
    src = rasterio.open(filepath)
    
    # Cell Information
    cell_columns = src.width
    cell_rows = src.height
    cell_data_type = src.dtypes[0]
    cell_x_size = src.transform.a 
    cell_y_size = abs(src.transform.e)
    
    # Bands
    num_bands = src.count
    band_data = {}
    for band in range(1, num_bands+1):
        dat = src.read(band)
        band_meta = dict(
            min_value = dat.min().item(),
            max_value = dat.max().item(),
            nan_value = src.nodatavals[band - 1],
            dtype = str(dat.dtype),
            dims = [
             {'name': 'x',
              'shape': dat.shape[0]
             },
             {'name': 'y',
              'shape': dat.shape[1]
             }, 
            ]
        )
        band_data[band] = band_meta
    
    # Extent
    extent_west, extent_south, extent_east, extent_north = src.bounds
    
    # Other metadata
    name = src.name.split('/')[1:][0]
    
    src.close()


    # Encode metadata
    box_str = f'{extent_south} {extent_west} {extent_north} {extent_east}'
    geo = base.GeoShape(box = box_str, validate_bbox=validate_bbox)

    crs = CRS.from_wkt(src.crs.wkt)

    srs = base.SpatialReference(
        name = crs.name,
        srsType=crs.type_name.split(' ')[0],
        code=crs.to_string(),
        wktString=crs.to_wkt()
    )
    
    place = base.Place(
        geo=geo,
        srs=srs
    )

    gridvariables = []
    for band_idx in band_data.keys():
        variable = datavariable.BandVariable(
            name = f'Band {band_idx}',
            dataType = band_data[band_idx]['dtype'],
            minValue = band_data[band_idx]['min_value'],
            maxValue = band_data[band_idx]['max_value'],
            noDataValue = band_data[band_idx]['nan_value'],
            band = band_idx,
            dimension=band_data[band_idx]['dims'],
        )
        gridvariables.append(variable)

    # get all file names that match the patter of the input filepath
    search_path = f"{'.'.join(filepath.split('.')[:-1])}.*"
    associated_files = glob(search_path)
    files = []
    for fpath in associated_files:
        files.append(
            base.MediaObject(
                contentUrl = f'https://hydroshare.org/my-resource/{fpath}',
                name = Path(fpath).name,
                sha256 = compute_sha256(Path(fpath)),
                contentSize = f'{os.path.getsize(Path(fpath))/1024} KB',
                encodingFormat = mimetypes.guess_type(Path(fpath))[0],
            )
        )

    r = raster.GeographicRaster(
        rows = cell_rows,
        columns = cell_columns,
        xCellSize = cell_x_size,
        yCellSize = cell_y_size,
        cellValueType = cell_data_type,
        variableMeasured = gridvariables,
        associatedMedia=files,
        spatialCoverage=place,
    )

    return r

### Encode a single band GeoTiff

In [9]:
meta = encode_raster_metadata('data/Onion3ad8o.tif', validate_bbox=True)
print(meta.model_dump_json(exclude_none=True, indent=4))

{
    "context": "https://hydroshare.org/schema",
    "type": "GeographicRaster",
    "spatialCoverage": {
        "type": "Place",
        "geo": {
            "type": "GeoShape",
            "box": "30.01462962962525 -98.30768518519098 30.27027777889825 -97.57953703383897"
        },
        "srs": {
            "type": "SpatialReference",
            "name": "NAD83",
            "srsType": "geographic",
            "code": "GEOGCS[\"NAD83\",DATUM[\"North_American_Datum_1983\",SPHEROID[\"GRS 1980\",6378137,298.257222101,AUTHORITY[\"EPSG\",\"7019\"]],AUTHORITY[\"EPSG\",\"6269\"]],PRIMEM[\"Greenwich\",0],UNIT[\"Degree\",0.0174532925199433],AXIS[\"Longitude\",EAST],AXIS[\"Latitude\",NORTH]]",
            "wktString": "GEOGCRS[\"NAD83\",DATUM[\"North American Datum 1983\",ELLIPSOID[\"GRS 1980\",6378137,298.257222101,LENGTHUNIT[\"metre\",1]],ID[\"EPSG\",6269]],PRIMEM[\"Greenwich\",0,ANGLEUNIT[\"Degree\",0.0174532925199433]],CS[ellipsoidal,2],AXIS[\"longitude\",east,ORDER[1],ANGLEUNIT[\"De

### Encode a multi-band GeoTiff

In [12]:
meta = encode_raster_metadata('data/landsat-multiband-sample.tif', validate_bbox=False)
print(meta.model_dump_json(exclude_none=True, indent=4))

{
    "context": "https://hydroshare.org/schema",
    "type": "GeographicRaster",
    "spatialCoverage": {
        "type": "Place",
        "geo": {
            "type": "GeoShape",
            "box": "4504185.0 254685.0 4743315.0 490215.0"
        },
        "srs": {
            "type": "SpatialReference",
            "name": "WGS 84 / UTM zone 15N",
            "srsType": "projected",
            "code": "EPSG:32615",
            "wktString": "PROJCRS[\"WGS 84 / UTM zone 15N\",BASEGEOGCRS[\"WGS 84\",DATUM[\"World Geodetic System 1984\",ELLIPSOID[\"WGS 84\",6378137,298.257223563,LENGTHUNIT[\"metre\",1]]],PRIMEM[\"Greenwich\",0,ANGLEUNIT[\"degree\",0.0174532925199433]],ID[\"EPSG\",4326]],CONVERSION[\"UTM zone 15N\",METHOD[\"Transverse Mercator\",ID[\"EPSG\",9807]],PARAMETER[\"Latitude of natural origin\",0,ANGLEUNIT[\"degree\",0.0174532925199433],ID[\"EPSG\",8801]],PARAMETER[\"Longitude of natural origin\",-93,ANGLEUNIT[\"degree\",0.0174532925199433],ID[\"EPSG\",8802]],PARAMETER[\"Scale

### Encode an ASCII Raster

In [13]:
meta = encode_raster_metadata('data/Onion3ad8o_ascii.asc')
print(meta.model_dump_json(exclude_none=True, indent=4))

{
    "context": "https://hydroshare.org/schema",
    "type": "GeographicRaster",
    "spatialCoverage": {
        "type": "Place",
        "geo": {
            "type": "GeoShape",
            "box": "30.014629629625 -98.307685185191 30.270277778898 -97.579537033839"
        },
        "srs": {
            "type": "SpatialReference",
            "name": "NAD83",
            "srsType": "geographic",
            "code": "GEOGCS[\"NAD83\",DATUM[\"North_American_Datum_1983\",SPHEROID[\"GRS 1980\",6378137,298.257222101,AUTHORITY[\"EPSG\",\"7019\"]],AUTHORITY[\"EPSG\",\"6269\"]],PRIMEM[\"Greenwich\",0],UNIT[\"Degree\",0.0174532925199433],AXIS[\"Longitude\",EAST],AXIS[\"Latitude\",NORTH]]",
            "wktString": "GEOGCRS[\"NAD83\",DATUM[\"North American Datum 1983\",ELLIPSOID[\"GRS 1980\",6378137,298.257222101,LENGTHUNIT[\"metre\",1]],ID[\"EPSG\",6269]],PRIMEM[\"Greenwich\",0,ANGLEUNIT[\"Degree\",0.0174532925199433]],CS[ellipsoidal,2],AXIS[\"longitude\",east,ORDER[1],ANGLEUNIT[\"Degree\",0